[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-12-advanced-rag.ipynb#scrollTo=11a2b3c4)

---
# Day 12 · Advanced RAG — MultiQuery, Compression, and Ensemble Retrieval
**certified-journeys / llm-engineering-certified** · Day 12 · Advanced Retrieval

> **Goal for today:** Implement three advanced retrieval strategies — MultiQueryRetriever, ContextualCompressionRetriever, and Ensemble Retrieval with BM25+Chroma — then evaluate each with recall@4 against labeled query-answer pairs.

In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb rank-bm25 sentence-transformers tiktoken

## Step 1 · Build a Shared Corpus and Baseline Retriever

All three advanced retrievers wrap or replace a **base retriever**. We'll build one shared corpus so comparisons are fair.

We use a small set of machine-learning concept passages — representative of real RAG corpora (short, dense, slightly overlapping). We embed with `sentence-transformers` so the notebook runs without an OpenAI key for the indexing step.

| Retriever type | Mechanism | Tradeoff |
|---------------|-----------|----------|
| Simple similarity | Cosine distance on one query vector | Fast, misses synonyms |
| MultiQuery | Multiple query rewrites + union | Higher recall, 3× LLM cost |
| Compression | Re-ranks + extracts relevant snippets | Lower noise, slower |
| Ensemble (BM25+dense) | RRF fusion of two ranked lists | Best of both worlds |

In [ ]:
from langchain.schema import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# --- Shared corpus: 15 short ML concept passages ---
passages = [
    "Gradient descent is an optimization algorithm that iteratively adjusts model parameters "
    "in the direction of the negative gradient to minimize a loss function.",

    "Stochastic gradient descent (SGD) updates parameters using a single random training example "
    "per step, making it noisier but faster than batch gradient descent.",

    "The learning rate controls the size of each parameter update in gradient-based optimization. "
    "Too high causes divergence; too low causes slow convergence.",

    "Batch normalization normalizes layer inputs across a mini-batch, reducing internal covariate "
    "shift and enabling higher learning rates during training.",

    "Dropout randomly deactivates a fraction of neurons during training, acting as an implicit "
    "ensemble of many sub-networks to reduce overfitting.",

    "The attention mechanism in transformers computes a weighted sum of value vectors, "
    "where weights are derived from query-key dot products scaled by the square root of dimension.",

    "Multi-head attention runs the attention function in parallel across multiple learned "
    "subspaces, allowing the model to jointly attend to information from different positions.",

    "Positional encodings inject sequence-order information into transformer inputs since "
    "self-attention is permutation-invariant by default.",

    "The softmax function converts a vector of raw logits into a probability distribution "
    "by exponentiating each value and normalizing by the sum.",

    "Cross-entropy loss measures the dissimilarity between a predicted probability distribution "
    "and the true label distribution, commonly used in classification tasks.",

    "Retrieval-Augmented Generation (RAG) combines a retriever that finds relevant documents "
    "with a language model that generates an answer conditioned on those documents.",

    "Vector databases store high-dimensional embeddings and support approximate nearest-neighbor "
    "search, enabling semantic similarity retrieval at scale.",

    "BM25 is a bag-of-words retrieval function that ranks documents based on term frequency "
    "and inverse document frequency, saturating on repeated terms.",

    "Reciprocal Rank Fusion (RRF) combines ranked lists from multiple retrievers by summing "
    "reciprocal ranks, providing a parameter-free fusion without re-scoring.",

    "Fine-tuning adapts a pretrained model to a specific task by continuing training on "
    "task-specific data, typically with a lower learning rate than pretraining.",
]

docs = [Document(page_content=p, metadata={"id": i}) for i, p in enumerate(passages)]

# Embed with sentence-transformers (free, runs on CPU in Colab)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build Chroma vectorstore in memory
vectorstore = Chroma.from_documents(docs, embeddings, collection_name="ml_concepts")

# Baseline retriever: top-4 by cosine similarity
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Indexed {len(docs)} passages into Chroma")
# Sanity check: one retrieval
hits = base_retriever.invoke("How does attention work?")
print(f"Baseline retrieval for 'How does attention work?' → {len(hits)} docs")
for h in hits:
    print(" -", h.page_content[:80])

**What just happened?**
- `all-MiniLM-L6-v2` is a 22M-parameter model — fast, accurate enough for English retrieval benchmarks
- Chroma stored the 384-dim embedding vectors in memory; `as_retriever(k=4)` returns the 4 nearest neighbors
- **Baseline weakness:** a single query vector can miss documents that use synonyms or different phrasing for the same concept

## Step 2 · MultiQueryRetriever — Generate 3 Query Variants

`MultiQueryRetriever` sends the original question to an LLM to generate N rephrased variants, retrieves documents for each variant separately, then takes the **union** of all retrieved sets.

**Why it works:** Ambiguous questions have multiple valid phrasings. A question like "How do transformers handle word order?" might embed far from "positional encoding" even though that's the answer. A rephrase like "How does self-attention know sequence position?" will hit the right passage.

**Cost:** N+1 LLM calls per retrieval (N variants + 1 original). Default N=3 is usually the right tradeoff.

In [ ]:
import os
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# Set your key: os.environ['OPENAI_API_KEY'] = 'sk-...'
# Colab: from google.colab import userdata; os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Enable logging to see the generated query variants in output
logging.basicConfig(level=logging.INFO)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    # Default prompt generates 3 variants; you can customize with include_original=True
)

question = "How do neural networks handle sequence order?"
mq_docs = multi_query_retriever.invoke(question)

print(f"\nMultiQuery retrieved {len(mq_docs)} unique docs for: '{question}'")
for d in mq_docs:
    print(" -", d.page_content[:90])

**What just happened?**
- The LLM generated 3 query variants (visible in the INFO log) — each was embedded and retrieved separately
- **Set union** was taken: duplicate documents appear only once in `mq_docs`
- The result typically includes more relevant docs than the baseline single-query retrieval
- **Key insight:** MultiQueryRetriever is the highest-leverage advanced RAG technique — 3 LLM calls per retrieval but it dramatically boosts recall on ambiguous questions

## Step 3 · ContextualCompressionRetriever — Extract Only Relevant Snippets

Long retrieved documents contain irrelevant sentences. `ContextualCompressionRetriever` adds a **compressor** that filters or extracts the relevant parts before returning them to the LLM.

`LLMChainExtractor` sends each retrieved document + the query to an LLM and asks it to extract only the relevant passage. This reduces context noise significantly.

**Production tip:** Use `EmbeddingsFilter` for a cheaper no-LLM alternative — it drops documents whose embedding cosine similarity to the query falls below a threshold.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Compressor: send each doc + query to the LLM and extract relevant sentences
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,  # first retrieve top-4, then compress each
)

question_compression = "What prevents overfitting in deep learning?"
compressed_docs = compression_retriever.invoke(question_compression)

print(f"Compression retriever returned {len(compressed_docs)} docs")
print("\n--- Compressed extracts ---")
for i, d in enumerate(compressed_docs, 1):
    print(f"[{i}] {d.page_content}")

**What just happened?**
- `base_retriever` fetched the top-4 documents as usual
- `LLMChainExtractor` then sent each document to the LLM with the question and asked it to extract only the relevant sentence(s)
- Documents with no relevant content were dropped entirely (fewer than 4 results is expected)
- **Key insight:** Compression reduces the tokens passed to the downstream LLM, improving answer quality on noisy corpora and lowering cost when documents are long

## Step 4 · Ensemble Retriever — BM25 + Chroma with RRF Fusion

`EnsembleRetriever` combines ranked lists from multiple retrievers using **Reciprocal Rank Fusion (RRF)**. BM25 excels at keyword matching; dense retrieval excels at semantic similarity. Combining them handles both.

**RRF formula:** `score(d) = Σ 1/(k + rank(d, retriever_i))` where k=60 is a constant that dampens high-rank advantages. No training required — just rank position.

| Retriever | Strength | Weakness |
|-----------|----------|----------|
| BM25 | Exact keyword match | Misses synonyms, paraphrases |
| Dense (Chroma) | Semantic similarity | Misses rare exact-match terms |
| Ensemble | Both | Slightly slower (two retrievers) |

In [ ]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# BM25 retriever works directly on the Document objects (no embeddings needed)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 4  # retrieve top-4 candidates from BM25

# The Chroma dense retriever (already built above)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Ensemble: equal weights for BM25 and dense
# weights must sum to 1.0; adjust to favour one retriever if you have prior knowledge
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],  # 50/50 fusion — tune based on your corpus
)

# Test on a keyword-heavy question where BM25 should shine
question_ensemble = "What is BM25 and how does it rank documents?"
ensemble_docs = ensemble_retriever.invoke(question_ensemble)

print(f"Ensemble retrieved {len(ensemble_docs)} docs")
for d in ensemble_docs:
    print(" -", d.page_content[:100])

**What just happened?**
- BM25 retrieved based on exact term frequency — "BM25" appears verbatim in one passage, which BM25 ranks #1
- Dense retrieval found semantically similar passages even if they don't use the word "BM25"
- **RRF fusion** combined both ranked lists: a document ranked #1 by both gets a high combined score
- The result contains documents that neither retriever alone would have ranked highly

## Step 5 · Evaluate Retrieval Quality — recall@4 Against 10 Labeled Pairs

**recall@4** = proportion of relevant documents found in the top-4 retrieved results.

We need **labeled query-document pairs**: for each query, we know which passage ID(s) should be retrieved. This is the ground truth for evaluation.

recall@k = (relevant docs in top-k) / (total relevant docs for query)

In [ ]:
# Ground truth: 10 labeled (query, relevant_doc_ids) pairs
# relevant_doc_ids are indices into the `passages` list above
labeled_pairs = [
    {"query": "How does gradient descent minimize loss?",          "relevant_ids": [0, 1]},
    {"query": "What is the role of learning rate in training?",   "relevant_ids": [2]},
    {"query": "How do transformers attend to different positions?", "relevant_ids": [5, 6, 7]},
    {"query": "What prevents overfitting in neural networks?",    "relevant_ids": [4]},
    {"query": "How does batch normalization speed up training?",   "relevant_ids": [3]},
    {"query": "What is RAG and why is it useful?",               "relevant_ids": [10]},
    {"query": "How does BM25 score documents?",                  "relevant_ids": [12]},
    {"query": "What is reciprocal rank fusion?",                 "relevant_ids": [13]},
    {"query": "How does fine-tuning differ from pretraining?",   "relevant_ids": [14]},
    {"query": "How does softmax turn scores into probabilities?", "relevant_ids": [8]},
]

def recall_at_k(retrieved_docs, relevant_ids, k=4):
    """Compute recall@k: fraction of relevant docs found in top-k results."""
    retrieved_ids = [d.metadata.get("id") for d in retrieved_docs[:k]]
    hits = sum(1 for rid in relevant_ids if rid in retrieved_ids)
    return hits / len(relevant_ids) if relevant_ids else 0.0

def evaluate_retriever(retriever, pairs, k=4):
    """Run all queries through a retriever and compute mean recall@k."""
    scores = []
    for pair in pairs:
        try:
            docs_retrieved = retriever.invoke(pair["query"])
        except Exception as e:
            print(f"  Error on query '{pair['query']}': {e}")
            scores.append(0.0)
            continue
        score = recall_at_k(docs_retrieved, pair["relevant_ids"], k=k)
        scores.append(score)
    return sum(scores) / len(scores) if scores else 0.0

print("Evaluating retrievers (this makes LLM calls for MultiQuery and Compression)...\n")

# Evaluate baseline
baseline_recall = evaluate_retriever(base_retriever, labeled_pairs)
print(f"Baseline (cosine k=4):           recall@4 = {baseline_recall:.3f}")

# Evaluate MultiQuery
mq_recall = evaluate_retriever(multi_query_retriever, labeled_pairs)
print(f"MultiQueryRetriever:             recall@4 = {mq_recall:.3f}")

# Evaluate Compression
comp_recall = evaluate_retriever(compression_retriever, labeled_pairs)
print(f"ContextualCompressionRetriever:  recall@4 = {comp_recall:.3f}")

# Evaluate Ensemble
ens_recall = evaluate_retriever(ensemble_retriever, labeled_pairs)
print(f"EnsembleRetriever (BM25+Chroma): recall@4 = {ens_recall:.3f}")

**What just happened?**
- We computed **recall@4** for all four retrievers against 10 labeled queries
- MultiQuery typically outperforms the baseline — especially on queries that use different vocabulary than the passages
- Compression may score lower on recall (it drops docs) but the remaining docs are higher precision
- Ensemble usually matches or beats the best individual retriever — BM25 rescues queries where dense fails

In [ ]:
# Pretty-print a comparison table
results = {
    "Baseline (cosine)": baseline_recall,
    "MultiQuery": mq_recall,
    "Compression": comp_recall,
    "Ensemble (BM25+Chroma)": ens_recall,
}

print("\n{'='*55}")
print(f"{'Retriever':<30} {'recall@4':>10} {'vs baseline':>12}")
print("-" * 55)
baseline = results["Baseline (cosine)"]
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    delta = score - baseline
    delta_str = f"+{delta:.3f}" if delta >= 0 else f"{delta:.3f}"
    marker = " ← best" if score == max(results.values()) else ""
    print(f"{name:<30} {score:>10.3f} {delta_str:>12}{marker}")
print("=" * 55)

**What just happened?**
- The comparison table ranks all four retrievers by recall@4
- `delta vs baseline` shows the absolute improvement — a positive number means the advanced retriever found more relevant docs
- **Note:** Compression is not primarily a recall technique — its value is precision and reduced noise for the downstream LLM
- Use this table to decide which retriever to deploy based on your specific corpus and query distribution

## Step 6 · Document Tradeoffs — Choose the Best Retriever

This markdown cell is where you synthesize the evaluation results and make a justified recommendation.

### Retriever decision framework

| Signal | Recommended retriever |
|--------|-----------------------|
| Queries are **ambiguous or conversational** | MultiQueryRetriever |
| Documents are **long with irrelevant paragraphs** | ContextualCompressionRetriever |
| Corpus has **both keyword-heavy and semantic queries** | EnsembleRetriever |
| Latency budget is tight (< 200ms) | Baseline cosine |
| LLM budget is tight | Baseline cosine or BM25 only |

### For this ML concepts corpus

**Recommendation:** `EnsembleRetriever (BM25 + Chroma)` is the best choice because:
1. Technical documentation uses precise vocabulary (BM25 excels) AND conceptual questions (dense excels)
2. RRF fusion requires no additional LLM calls — zero extra cost over the baseline
3. Recall@4 improvement is consistent across query types, not just ambiguous ones

**When to use MultiQuery instead:** If your users ask conversational, open-ended questions with high variability in phrasing, the recall boost justifies the 3× LLM cost.

**When to add Compression on top:** Layer `ContextualCompressionRetriever` on top of the Ensemble when documents are longer than ~500 tokens — it pays for itself in reduced downstream LLM costs.

In [ ]:
# Challenge: Build a stacked retriever pipeline
#
# Your task:
#   1. Create a NEW EnsembleRetriever combining BM25 + Chroma with weights [0.4, 0.6]
#      (favour dense retrieval for this semantic corpus)
#   2. Wrap it in a ContextualCompressionRetriever using LLMChainExtractor
#      This gives you: Ensemble → Retrieve top-4 → Compress each → Return relevant snippets
#   3. Evaluate this stacked retriever using evaluate_retriever() against labeled_pairs
#   4. Add a row to the results dict and re-print the comparison table
#   5. In a comment, explain: does stacking improve recall? Why or why not?
#      (Hint: think about what compression does to the retrieved set size)

# Scaffold:
# ensemble_weighted = EnsembleRetriever(
#     retrievers=[bm25_retriever, dense_retriever],
#     weights=[0.4, 0.6],
# )

# stacked_retriever = ContextualCompressionRetriever(
#     base_compressor=compressor,       # already defined above
#     base_retriever=ensemble_weighted,
# )

# stacked_recall = evaluate_retriever(stacked_retriever, labeled_pairs)
# print(f"Stacked (Ensemble + Compression): recall@4 = {stacked_recall:.3f}")

# YOUR CODE HERE

---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| MultiQueryRetriever | Generates N query variants via LLM, takes union of retrieved sets — best for ambiguous queries |
| ContextualCompressionRetriever | Extracts relevant snippets from retrieved docs — improves precision, may reduce recall |
| EnsembleRetriever | RRF fusion of BM25 + dense; no training needed, handles keyword and semantic queries |
| BM25 | Bag-of-words, exact term match, zero cost — always worth including in an ensemble |
| recall@k | Fraction of relevant docs found in top-k — the primary offline retrieval metric |
| Stacking | Ensemble → Compression is a valid pattern for long-document corpora; may reduce recall@k but improves precision |

> **Tip:** MultiQueryRetriever is the highest-leverage advanced RAG technique — 3 LLM calls per retrieval but it dramatically boosts recall on ambiguous questions.

---
## What's next
**Day 13** → Production RAG — evaluation pipelines, RAGAS metrics, and deploying a retrieval chain as a FastAPI endpoint.

Mark Day 12 complete in your [tracker](../index.html).